# requires-grad-propagation — worked example 2: Verify requires_grad Propagation Against Real PyTorch Operations

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-propagation`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

PyTorch itself follows the same three-gate rule when computing `requires_grad` for operation outputs. You can verify this empirically: `a + b` gets `requires_grad=True` if either `a` or `b` does (and global tracking is on). Inside `torch.no_grad()`, the same addition produces `requires_grad=False`. Understanding this lets you predict whether any op's output will carry a gradient.

## Worked solution

**Step 1 — set up leaf tensors.** Create `a` with `requires_grad=True` and `b` without it.

**Step 2 — test outside `no_grad`.** `c = a + b` should have `requires_grad=True` because `a` does, the op is differentiable, and tracking is on.

**Step 3 — test inside `no_grad`.** Wrapping in `with t.no_grad()` turns off the global tracking gate. Now `c = a + b` has `requires_grad=False`.

**Step 4 — test with both inputs grad-free.** Even with tracking on and op differentiable, if neither input has `requires_grad`, the output won't either.

**Step 5 — confirm with our function.** Run the same cases through `propagate_requires_grad` and verify the outputs match PyTorch's actual behavior.

In [ ]:
import torch as t

def propagate_requires_grad(args, is_differentiable, grad_tracking_enabled):
    return (
        grad_tracking_enabled
        and is_differentiable
        and any(isinstance(a, t.Tensor) and a.requires_grad for a in args)
    )

# --- exercise and print ---
a = t.tensor([3.0], requires_grad=True)
b = t.tensor([5.0])  # no grad

# Case 1: normal addition, tracking on
c1 = a + b
our1 = propagate_requires_grad((a, b), is_differentiable=True, grad_tracking_enabled=True)
print(f'Case 1 — add with tracking:  PyTorch={c1.requires_grad}, ours={our1}, match={c1.requires_grad==our1}')

# Case 2: addition inside no_grad
with t.no_grad():
    c2 = a + b
our2 = propagate_requires_grad((a, b), is_differentiable=True, grad_tracking_enabled=False)
print(f'Case 2 — add in no_grad:     PyTorch={c2.requires_grad}, ours={our2}, match={c2.requires_grad==our2}')

# Case 3: neither input has grad
c3 = b + b
our3 = propagate_requires_grad((b, b), is_differentiable=True, grad_tracking_enabled=True)
print(f'Case 3 — both no grad:       PyTorch={c3.requires_grad}, ours={our3}, match={c3.requires_grad==our3}')

# Case 4: non-differentiable op (simulated)
our4 = propagate_requires_grad((a, b), is_differentiable=False, grad_tracking_enabled=True)
print(f'Case 4 — non-differentiable: ours={our4} (expect False)')